# 01 — Исследование датасета ForteBank

Проверяем видео, разметку, обычные действия и возможные подсказки для модели до обучения.
Исходная разметка — `data/annotations/events.csv`: ноутбук её **не перезаписывает**.
Расчётные таблицы и предложение разбиения сохраняются в `data/processed/`, изображения — в `reports/figures/`.

Видео ищутся в `open/`, `data/open/`, `data/raw/open/`, `data/raw/videos/`, `data/raw/`.
Путь можно задать явно через `VIDEO_DIR`. Некорректный явный путь вызывает ошибку.
`confidence` в разметке — оценка гипотезы человеком, а не вероятность модели.
Описания и нормативные ссылки требуют проверки; исходные метки не меняются автоматически.

In [ ]:
import hashlib
import json
import os
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib.patches import Rectangle
from tqdm.auto import tqdm

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "src" / "overfitted_underqualified").is_dir() and (p / "notebooks").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Запустите ноутбук из репозитория или notebooks/")
sys.path.insert(0, str(ROOT / "src"))
from overfitted_underqualified.eda import (
    LABELS,
    build_groups,
    clip_id_from_name,
    clip_sort_key,
    inspect_video,
    parse_timecode,
    propose_split,
    read_frames,
    scene_candidates,
    validate_dataset,
    window_timestamps,
)

RANDOM_STATE = 42
VIDEO_EXTENSIONS = {".mov", ".mp4", ".avi", ".mkv", ".m4v"}
WINDOW_SEC, FRAME_STEP_SEC = 4.0, 0.5
SCENE_DISTANCE_THRESHOLD = 0.12  # Эвристика; пары обязательно просмотреть.
ANNOTATIONS_DIR = ROOT / "data" / "annotations"
PROCESSED_DIR = ROOT / "data" / "processed"
FIGURES_DIR = ROOT / "reports" / "figures"
EVENT_FRAMES_DIR = FIGURES_DIR / "event_frames"
NORMAL_FRAMES_DIR = FIGURES_DIR / "normal_frames"
EVENTS_PATH = ANNOTATIONS_DIR / "events.csv"
SCENES_PATH = ANNOTATIONS_DIR / "scenes.csv"
ZONES_PATH = ANNOTATIONS_DIR / "zones.json"
for folder in (PROCESSED_DIR, FIGURES_DIR, EVENT_FRAMES_DIR, NORMAL_FRAMES_DIR):
    folder.mkdir(parents=True, exist_ok=True)

def collect_videos(folder):
    return sorted((p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS),
                  key=lambda p: (clip_sort_key(clip_id_from_name(p.name)), p.as_posix()))

explicit = os.environ.get("VIDEO_DIR") or os.environ.get("FORTEBANK_OPEN_DIR")
if explicit:
    DATA_DIR = Path(explicit).expanduser().resolve()
    if not DATA_DIR.is_dir() or not collect_videos(DATA_DIR):
        raise FileNotFoundError(f"В указанном каталоге нет видео: {DATA_DIR}")
else:
    candidates = [ROOT / p for p in ("open", "data/open", "data/raw/open", "data/raw/videos", "data/raw")]
    DATA_DIR = next((p for p in candidates if p.is_dir() and collect_videos(p)), None)
    if DATA_DIR is None:
        raise FileNotFoundError("Видео не найдены. Добавьте их в open/ или задайте VIDEO_DIR.")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 60)

def save_figure(fig, filename):
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / filename, dpi=120, bbox_inches="tight")
    plt.show()
    plt.close(fig)

print(f"Видео: {DATA_DIR}")
print(f"OpenCV {cv2.__version__}; результаты: {PROCESSED_DIR}")


## 1. Техническая проверка и точные копии

Для каждого файла считаем SHA-256, проверяем метаданные и декодируем пять кадров от начала до 98% длительности.
`readable` означает, что метаданные корректны и прочитались все пять проб. Это выборочная проверка,
а не гарантия целостности каждого кадра. Последний расчётный кадр проверяется отдельно: ошибка
позиционирования у конца MOV даёт предупреждение, а не объявляет весь ролик нечитаемым.
Длительность считается как число кадров / FPS;
для переменной частоты кадров она может быть приблизительной.

In [ ]:
video_paths = collect_videos(DATA_DIR)
records, thumbnails = [], {}
for path in tqdm(video_paths, desc="video checks"):
    record, sampled = inspect_video(path, DATA_DIR)
    records.append(record)
    # Не выбираем случайный файл при одинаковом идентификаторе.
    thumbnails.setdefault(record["clip_id"], sampled)
inventory = pd.DataFrame(records)
inventory.to_csv(PROCESSED_DIR / "video_inventory.csv", index=False)
readable = inventory.loc[inventory["readable"]].copy()
print(f"Файлов: {len(inventory)}, прошли выборочное чтение: {len(readable)}")
display(inventory)
display(inventory.loc[inventory["clip_id"].duplicated(False), ["clip_id", "video_rel"]])
display(inventory.loc[inventory["sha256"].ne("") & inventory["sha256"].duplicated(False), ["clip_id", "video_rel", "sha256"]])


## 2. Разметка и отчёт об ошибках

Таймкод должен быть корректным и попадать внутрь ролика. Для нарушения он обязателен.
Повторяющиеся ID, конфликтующие метки точных копий, отсутствующие файлы и ошибки чтения
попадают в `validation_issues.csv`. Проблемные ролики исключаются из дальнейшего анализа и разбиения;
ошибки остаются видны в отчёте и итоговом чеклисте.

In [ ]:
if not EVENTS_PATH.is_file():
    raise FileNotFoundError(f"Нет исходной разметки: {EVENTS_PATH}")
source_digest = hashlib.sha256(EVENTS_PATH.read_bytes()).hexdigest()
events = pd.read_csv(EVENTS_PATH, dtype={"clip_id": "string", "timecode": "string"})
required = {"clip_id", "timecode", "source_label", "observed_event", "observed_event_type", "candidate_violation_type"}
if missing := required - set(events.columns):
    raise ValueError(f"В events.csv отсутствуют колонки: {sorted(missing)}")
for column in ("confidence", "legal_basis", "review_status"):
    if column not in events:
        events[column] = pd.Series(pd.NA, index=events.index, dtype="string")
events["timecode_sec"] = events["timecode"].map(parse_timecode)
issues = validate_dataset(events, inventory)
issues.to_csv(PROCESSED_DIR / "validation_issues.csv", index=False)
display(issues)
blocked = set(issues.loc[issues["severity"].eq("error"), "clip_id"])
eligible = events.loc[~events["clip_id"].astype(str).isin(blocked)].copy()
technical = inventory.loc[~inventory["clip_id"].duplicated(False)].rename(columns={"video_rel": "video_file"})
# Расчётные поля заменяем актуальными значениями без изменения events.csv.
derived = [c for c in technical.columns if c != "clip_id"]
analysis = eligible.drop(columns=derived, errors="ignore").merge(technical, on="clip_id", validate="one_to_one")
analysis = analysis.sort_values("clip_id", key=lambda s: s.map(clip_sort_key)).reset_index(drop=True)
analysis.to_csv(PROCESSED_DIR / "events_enriched.csv", index=False)
print(f"Размечено: {len(events)}; пригодно для анализа: {len(analysis)}; ошибок: {issues['severity'].eq('error').sum()}")
display(events)


## 3. Длительность, FPS, разрешение и баланс меток

Графики строятся заново при каждом запуске. Проверяем оба класса, редкие типы событий
и относительное положение события внутри ролика. Данные исключённых файлов остаются в отчёте ошибок.

In [ ]:
generated_figures = set()
if not readable.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].hist(readable["duration_sec"], bins=12)
    axes[0].set(xlabel="Длительность, сек", ylabel="Видео")
    readable["fps"].round(2).value_counts().sort_index().plot.bar(ax=axes[1], title="FPS")
    resolution = readable["width"].astype(str) + "×" + readable["height"].astype(str)
    resolution.value_counts().plot.bar(ax=axes[2], title="Разрешение")
    save_figure(fig, "video_parameters.png")
    generated_figures.add("video_parameters.png")
    display(readable[["duration_sec", "fps", "width", "height"]].describe())

label_counts = events["source_label"].value_counts().reindex(LABELS, fill_value=0)
fig, ax = plt.subplots(figsize=(6, 4))
label_counts.plot.bar(ax=ax, ylabel="Число роликов", title="Исходные метки")
save_figure(fig, "label_distribution.png")
generated_figures.add("label_distribution.png")
print(label_counts.to_string())
if label_counts.min() > 0:
    print(f"Отношение большего класса к меньшему: {label_counts.max() / label_counts.min():.2f}")
else:
    print("Один из классов отсутствует; обучение и валидация бинарной модели пока невозможны.")

categories = events.loc[events["source_label"].eq("violation"), "candidate_violation_type"].dropna().value_counts()
if not categories.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    categories.sort_values().plot.barh(ax=ax, xlabel="Число роликов", title="Гипотезы категорий")
    save_figure(fig, "candidate_violation_types.png")
    generated_figures.add("candidate_violation_types.png")
types = events.loc[events["source_label"].eq("violation"), "observed_event_type"].value_counts()
print(f"Типов событий: {len(types)}; с единственным примером: {int(types.eq(1).sum())}")
timed = analysis.loc[analysis["source_label"].eq("violation") & analysis["timecode_sec"].notna()]
if not timed.empty:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].hist(timed["timecode_sec"], bins=10)
    axes[0].set(xlabel="Таймкод, сек", ylabel="События")
    axes[1].hist(timed["timecode_sec"] / timed["duration_sec"], bins=10, range=(0, 1))
    axes[1].set(xlabel="Положение в ролике (0 — начало, 1 — конец)")
    save_figure(fig, "timecode_distribution.png")
    generated_figures.add("timecode_distribution.png")


## 4. Визуальный просмотр обоих классов

Для нарушения берём окно ±4 секунды с шагом 0,5 секунды, обязательно включая сам таймкод.
Кадры не выходят за последнюю позицию видео. Для обычного ролика берём пять кадров по всей длительности.
Это обзор: короткое действие всё ещё может требовать просмотра исходного видео с полной частотой кадров.
Показываются по два контактных листа каждого класса, остальные сохраняются в папки `event_frames/` и `normal_frames/`.
`frame_extraction.csv` фиксирует каждую попытку, число успешных кадров и полноту последовательности.

Обычные действия тоже нужно описывать: передача денег и уход от стола могут встречаться в обоих классах.

In [ ]:
def save_contact_sheet(clip_id, path, stamps, folder, show=False):
    frames = read_frames(path, stamps)
    columns = min(5, len(stamps))
    rows = int(np.ceil(len(stamps) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(3 * columns, 2.5 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, stamp, frame in zip(axes.flat, stamps, frames):
        if frame is not None:
            ax.imshow(frame)
        else:
            ax.text(0.5, 0.5, "Кадр не прочитан", ha="center", transform=ax.transAxes)
        ax.set_title(f"{stamp:.2f} s")
    fig.suptitle(clip_id)
    target = folder / f"{clip_id}.png"
    fig.tight_layout()
    fig.savefig(target, dpi=90, bbox_inches="tight")
    if show:
        plt.show()
    plt.close(fig)
    return {"clip_id": clip_id, "expected": len(stamps), "decoded": sum(f is not None for f in frames),
            "complete": all(f is not None for f in frames), "output": str(target.relative_to(ROOT))}

frame_records, shown = [], {label: 0 for label in LABELS}
for row in tqdm(list(analysis.itertuples(index=False)), desc="contact sheets"):
    if row.source_label == "violation":
        stamps = window_timestamps(row.timecode_sec, row.duration_sec, row.fps, WINDOW_SEC, FRAME_STEP_SEC)
        folder = EVENT_FRAMES_DIR
    else:
        stamps = np.linspace(0, max(0, min(row.duration_sec - 1 / row.fps, row.duration_sec * 0.98)), 5).tolist()
        folder = NORMAL_FRAMES_DIR
    record = save_contact_sheet(row.clip_id, DATA_DIR / row.video_file, stamps, folder, shown[row.source_label] < 2)
    record["source_label"] = row.source_label
    frame_records.append(record)
    shown[row.source_label] += 1
frame_report = pd.DataFrame(frame_records, columns=["clip_id", "expected", "decoded", "complete", "output", "source_label"])
frame_report.to_csv(PROCESSED_DIR / "frame_extraction.csv", index=False)
display(frame_report)


## 5. Возможные подсказки для модели

Сравниваем технические параметры, яркость и резкость между классами. Эти показатели не доказывают
причинную связь, но помогают заметить ситуацию, когда класс угадывается по длине ролика или освещению.
Резкость зависит и от содержания кадра; универсальный порог «плохого видео» здесь не задаём.
Фон и обстановку проверяем также по парам похожих сцен в следующем разделе.

In [ ]:
if not analysis.empty:
    summary = analysis.groupby("source_label")[["duration_sec", "fps", "brightness", "blur_score"]].agg(["count", "median", "min", "max"])
    summary.to_csv(PROCESSED_DIR / "class_parameter_summary.csv")
    display(summary)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, column in zip(axes, ["duration_sec", "brightness", "blur_score"]):
        sns.boxplot(data=analysis, x="source_label", y=column, ax=ax)
        sns.stripplot(data=analysis, x="source_label", y=column, ax=ax, color="black", size=3)
    save_figure(fig, "parameters_by_label.png")
    generated_figures.add("parameters_by_label.png")
    for column in ("fps", "width", "height"):
        print(column)
        display(pd.crosstab(analysis[column], analysis["source_label"]))


## 6. Похожие сцены и предложение разделения

Точные копии объединяются по SHA-256. Для поиска похожей обстановки сравниваем уменьшенные
медианные изображения пяти кадров, без использования меток. Это эвристика: возможны лишние пары
и пропуски. `scene_candidates.csv` и обзор кадров помогают вручную проверить фон, камеру и помещение.

Необязательный `data/annotations/scenes.csv` содержит `clip_id,camera_id,session_id,scene_group,reviewed`.
Используйте одинаковый camera_id для одной камеры; session_id должен быть уникальным идентификатором
сеанса съёмки. `reviewed=true` означает, что человек проверил принадлежность ролика группе.
При отсутствии файла шаблон создаётся **в processed/** и не считается подтверждённой разметкой.

В одну группу объединяются точные копии, визуальные кандидаты и общие camera_id/session_id/scene_group.
Группа целиком попадает в одну часть. Если независимых групп недостаточно, разбиение не создаётся:
покадровой или случайной замены нет. Разбиение без проверки сцен помечается `provisional`.

In [ ]:
usable_ids = set(analysis["clip_id"])
pairs = scene_candidates({k: v for k, v in thumbnails.items() if k in usable_ids}, SCENE_DISTANCE_THRESHOLD)
pairs.to_csv(PROCESSED_DIR / "scene_candidates.csv", index=False)
display(pairs)
scene_columns = ["clip_id", "camera_id", "session_id", "scene_group", "reviewed"]
template = analysis[["clip_id"]].copy()
for column in scene_columns[1:]:
    template[column] = "false" if column == "reviewed" else ""
template.to_csv(PROCESSED_DIR / "scenes_template.csv", index=False)
scenes = pd.read_csv(SCENES_PATH, dtype="string").fillna("") if SCENES_PATH.exists() else template
if not set(scene_columns).issubset(scenes.columns) or scenes["clip_id"].duplicated().any():
    raise ValueError("scenes.csv: нужны уникальные clip_id и колонки " + ", ".join(scene_columns))
reviewed = scenes["reviewed"].astype(str).str.lower().eq("true")
assigned = scenes[["camera_id", "session_id", "scene_group"]].apply(lambda c: c.astype(str).str.strip().ne("")).any(axis=1)
scene_review_complete = usable_ids <= set(scenes.loc[reviewed & assigned, "clip_id"]) and bool(usable_ids)
display(scenes)
if not pairs.empty:
    preview = pairs.head(6)
    fig, axes = plt.subplots(len(preview), 2, figsize=(10, 3 * len(preview)), squeeze=False)
    for row_axes, pair in zip(axes, preview.itertuples(index=False)):
        for ax, clip_id in zip(row_axes, (pair.clip_a, pair.clip_b)):
            ax.imshow(np.median(np.stack(thumbnails[clip_id]), axis=0), cmap="gray", vmin=0, vmax=255)
            ax.set_title(f"{clip_id}; distance={pair.scene_distance:.3f}")
            ax.axis("off")
    save_figure(fig, "scene_candidates_preview.png")
groups = build_groups(analysis, inventory, pairs, scenes)
groups.to_csv(PROCESSED_DIR / "scene_groups.csv", index=False)
display(groups.groupby("group_id")["clip_id"].agg(list))
split, split_status = propose_split(analysis, groups, RANDOM_STATE)
split_path = PROCESSED_DIR / "split_proposal.csv"
if not split.empty:
    split["status"] = "reviewed_groups" if scene_review_complete else "provisional"
    split.to_csv(split_path, index=False)
    display(pd.crosstab(split["split"], split["source_label"]))
    print(split_status if not scene_review_complete else "Группы проверены; разбиение сохранено")
else:
    # Не оставляем старое предложение, которое могло стать некорректным.
    pd.DataFrame(columns=["clip_id", "source_label", "group_id", "split", "status"]).to_csv(split_path, index=False)
    print("Разбиение недоступно:", split_status)


## 7. Зоны кадра из отдельного файла

Зоны при необходимости задаются в `data/annotations/zones.json` как
`{"clip_id": {"zone_name": [x1, y1, x2, y2]}}`. Координаты проверяются по размеру кадра.
Без файла этот дополнительный этап пропускается; координаты не вшиты в ноутбук.

In [ ]:
zones = json.loads(ZONES_PATH.read_text(encoding="utf-8")) if ZONES_PATH.exists() else {}
for clip_id, clip_zones in zones.items():
    match = analysis.loc[analysis["clip_id"].eq(clip_id)]
    if match.empty:
        print(f"{clip_id}: нет пригодного видео")
        continue
    row = match.iloc[0]
    stamp = row["timecode_sec"] if pd.notna(row["timecode_sec"]) else row["duration_sec"] / 2
    frame = read_frames(DATA_DIR / row["video_file"], [min(stamp, row["duration_sec"] - 1 / row["fps"])])[0]
    if frame is None:
        print(f"{clip_id}: кадр не прочитан")
        continue
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.imshow(frame)
    for name, box in clip_zones.items():
        if len(box) != 4 or not all(isinstance(v, (int, float)) and np.isfinite(v) for v in box):
            raise ValueError(f"Некорректная зона {clip_id}/{name}")
        x1, y1, x2, y2 = box
        if not (0 <= x1 < x2 <= frame.shape[1] and 0 <= y1 < y2 <= frame.shape[0]):
            raise ValueError(f"Зона вне кадра: {clip_id}/{name}")
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="red"))
        ax.text(x1, y1, name, color="red")
    ax.axis("off")
    save_figure(fig, f"zones_{clip_id}.png")
if not zones:
    print("Зоны пока не размечены — дополнительный этап пропущен.")


## 8. Итог проверки текущего запуска

Чеклист опирается на идентификаторы и результаты этого запуска, а не на старые файлы графиков.
Сохранение контактного листа не означает ручную проверку. Предложение разбиения требует проверки сцен.
Все результаты относятся к текущим видео и SHA-256 исходного CSV в `eda_run.json`.

In [ ]:
annotation_ids = set(events["clip_id"].dropna())
readable_ids = set(readable["clip_id"])
all_frames_complete = (set(frame_report["clip_id"]) == annotation_ids and not frame_report.empty
                       and frame_report["complete"].all())
source_unchanged = hashlib.sha256(EVENTS_PATH.read_bytes()).hexdigest() == source_digest
checks = {
    "Нет ошибок видео и разметки": not issues["severity"].eq("error").any(),
    "Все размеченные видео найдены и прошли чтение": bool(annotation_ids) and annotation_ids <= readable_ids
        and not inventory["clip_id"].duplicated().any(),
    "Графики параметров и классов построены в этом запуске":
        {"video_parameters.png", "label_distribution.png", "parameters_by_label.png"} <= generated_figures,
    "Все ожидаемые кадры обоих классов прочитаны": bool(all_frames_complete),
    "Группы сцен проверены человеком": scene_review_complete,
    "Предложение разбиения создано": not split.empty,
    "Исходная разметка не изменилась": source_unchanged,
}
status = pd.DataFrame({"Проверка": list(checks), "Готово": list(checks.values())})
status.to_csv(PROCESSED_DIR / "eda_checklist.csv", index=False)
manifest = {"annotations_sha256": source_digest, "video_directory": str(DATA_DIR),
            "video_sha256": dict(zip(inventory["video_rel"], inventory["sha256"])),
            "scene_distance_threshold": SCENE_DISTANCE_THRESHOLD, "seed": RANDOM_STATE,
            "split_status": split_status, "checks": {key: bool(value) for key, value in checks.items()}}
(PROCESSED_DIR / "eda_run.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
display(status)
assert source_unchanged, "Исходный CSV изменился во время запуска; перезапустите анализ"
print("Для обучения используйте events_enriched.csv и проверенное split_proposal.csv.")


## Дальнейшие действия

Сначала исправьте ошибки из `validation_issues.csv`, просмотрите обычные и отмеченные действия,
уточните описания событий и заполните группы сцен. Затем перезапустите ноутбук.
Проверяйте модель на целых группах видео; используйте одно разбиение для сравнения экспериментов.
Если вся выборка снята одной камерой, независимая проверка по новой камере требует дополнительных данных.
Разметка события в одной точке не задаёт длительность нарушения: соседние кадры нельзя автоматически
считать положительными примерами. Для обучения по отрезкам потребуется разметить начало и конец действия.